In [ ]:
import os
import glob
import json
import random
import datetime
import re
import cv2
import math
from typing import List, Optional, Dict, Any, Tuple

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from captum.concept import TCAV, Concept
from captum.concept._utils.classifier import Classifier
from torch.utils.data import Dataset, DataLoader, IterableDataset, TensorDataset
from pathlib import Path
from PIL import Image

from scipy.stats import ttest_ind

from captum.concept import TCAV, Concept, Classifier
from captum.attr import LayerGradientXActivation, LayerIntegratedGradients
from captum.concept._utils.data_iterator import dataset_to_dataloader, CustomIterableDataset
from captum.concept._utils.common import concepts_to_str


from ...model.model_sq import TNet
from ...tools import plotting, metrics, model_io, tcav, data, utils
from ...tools.plotting import plot_concepts
from ...model.model_dylan import FlexibleTNet  



In [ ]:
PESOS = "../../../saved_models/model_conv1_fc1_best.pt"
DEVICE = torch.device('cpu')

In [ ]:
def transform(img):
    return transforms.Compose(
        [
            transforms.Resize((128, 128), antialias=True),
            transforms.ToTensor()
        ]
    )(img)

def get_tensor_from_filename(filename):
    img = Image.open(filename).convert("RGB")
    return transform(img)

def load_image_tensors(class_name, root_path='../../../data/tcav/image/imagenet/', transform=True):
    path = os.path.join(root_path, class_name)
    filenames = glob.glob(path + '/*.jpg')

    tensors = []
    for filename in filenames:
        img = Image.open(filename).convert('RGB')
        tensors.append(transform(img) if transform else img)
    
    return tensors

def assemble_concept(name, id, concepts_path="data/tcav/image/concepts/"):
    concept_path = os.path.join(concepts_path, name) + "/"
    dataset = CustomIterableDataset(get_tensor_from_filename, concept_path)
    concept_iter = dataset_to_dataloader(dataset)

    return Concept(id=id, name=name, data_iter=concept_iter)
